In [ ]:
# Cell 1: Day 2 Setup
!pip install -q torch transformers peft bitsandbytes datasets accelerate
!pip install -q wandb tqdm

import torch
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ NO GPU DETECTED - Go to Runtime → Change runtime type → T4 GPU")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 17.1 MB/s eta 0:00:00
✅ PyTorch version: 2.9.0+cu126
✅ CUDA available: True
✅ GPU: Tesla T4
✅ GPU Memory: 15.8 GB


In [ ]:
# Cell 2: Login to Weights & Biases
import wandb

# Get your API key from: https://wandb.ai/authorize
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 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


wandb: WARNING Invalid choice
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aravind-b25 (aravind-b25-northeastern-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
# Cell 3: Load Dataset
from datasets import load_from_disk

# If you're continuing in the same notebook, data should be there
# If new notebook, we'll need to recreate it

import os
if os.path.exists('data/processed/sql_dataset'):
    dataset = load_from_disk('data/processed/sql_dataset')
    print("✅ Dataset loaded from disk!")
else:
    print("⚠️ Dataset not found - we'll recreate it quickly")

    from datasets import load_dataset, Dataset, DatasetDict
    from sklearn.model_selection import train_test_split

    # Load Spider
    spider = load_dataset("spider")

    def format_example(example):
        system_prompt = """You are an expert SQL assistant. Given a database schema and a natural language question, generate the correct SQL query. Only output the SQL query, nothing else."""
        prompt = f"""<s>[INST] {system_prompt}

Database: {example['db_id']}

Question: {example['question']} [/INST]
{example['query']}</s>"""
        return {'text': prompt, 'question': example['question'], 'query': example['query'], 'db_id': example['db_id']}

    train_formatted = [format_example(ex) for ex in spider['train']]
    val_data = list(spider['validation'])
    val_split, test_split = train_test_split(val_data, test_size=0.5, random_state=42)
    val_formatted = [format_example(ex) for ex in val_split]
    test_formatted = [format_example(ex) for ex in test_split]

    dataset = DatasetDict({
        'train': Dataset.from_list(train_formatted),
        'validation': Dataset.from_list(val_formatted),
        'test': Dataset.from_list(test_formatted)
    })

    os.makedirs('data/processed', exist_ok=True)
    dataset.save_to_disk('data/processed/sql_dataset')
    print("✅ Dataset recreated and saved!")

print(f"\n📊 Dataset Summary:")
print(f"   Train: {len(dataset['train'])} samples")
print(f"   Validation: {len(dataset['validation'])} samples")
print(f"   Test: {len(dataset['test'])} samples")

⚠️ Dataset not found - we'll recreate it quickly


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

spider/train-00000-of-00001.parquet:   0%|          | 0.00/831k [00:00<?, ?B/s]

spider/validation-00000-of-00001.parquet:   0%|          | 0.00/126k [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/517 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/517 [00:00<?, ? examples/s]

✅ Dataset recreated and saved!

📊 Dataset Summary:
   Train: 7000 samples
   Validation: 517 samples
   Test: 517 samples


In [ ]:
# Cell 4: Load Model with QLoRA
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

print("🔄 Loading Mistral-7B with 4-bit quantization...")
print("   (This takes 2-3 minutes - be patient!)\n")

# Model name
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

# 4-bit quantization config (reduces memory from 28GB to ~5GB)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print(f"✅ Model loaded: {MODEL_NAME}")
print(f"✅ Model memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

🔄 Loading Mistral-7B with 4-bit quantization...
   (This takes 2-3 minutes - be patient!)



config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

✅ Model loaded: mistralai/Mistral-7B-Instruct-v0.2
✅ Model memory footprint: 4.01 GB


In [ ]:
# Cell 5: Configure LoRA
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for training
model = prepare_model_for_kbit_training(model)

# LoRA configuration (we'll test 3 different configs)
lora_config = LoraConfig(
    r=16,                          # Rank - we'll vary this
    lora_alpha=32,                 # Alpha - typically 2x rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Attention layers
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

trainable params: 13,631,488 || all params: 7,255,363,584 || trainable%: 0.1879


In [ ]:
# Cell 6: Tokenize Dataset
from transformers import DataCollatorForLanguageModeling

def tokenize_function(examples):
    result = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )
    result["labels"] = result["input_ids"].copy()
    return result

print("🔄 Tokenizing dataset...")

# Tokenize all splits
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing"
)

print(f"\n✅ Tokenization complete!")
print(f"   Train: {len(tokenized_dataset['train'])} samples")
print(f"   Validation: {len(tokenized_dataset['validation'])} samples")

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # We're doing causal LM, not masked LM
)

print("✅ Data collator ready!")

🔄 Tokenizing dataset...


Tokenizing:   0%|          | 0/7000 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/517 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/517 [00:00<?, ? examples/s]


✅ Tokenization complete!
   Train: 7000 samples
   Validation: 517 samples
✅ Data collator ready!


In [ ]:
# Cell 7: Training Arguments
from transformers import TrainingArguments

# Config B: lr=1e-4, r=16 (already set), epochs=3
training_args = TrainingArguments(
    output_dir="./models/config_b",

    # Training params
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,  # Effective batch size = 4 * 4 = 16

    # Optimizer
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    optim="paged_adamw_8bit",

    # Logging (for W&B)
    logging_steps=25,
    logging_first_step=True,
    report_to="wandb",
    run_name="sql-rag-config-b",

    # Evaluation
    eval_strategy="steps",
    eval_steps=100,

    # Checkpointing
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # Performance
    fp16=True,
    gradient_checkpointing=True,

    # Misc
    remove_unused_columns=False,
    label_names=["labels"]
)

print("✅ Training arguments configured!")
print(f"\n📋 Config B Settings:")
print(f"   Learning rate: {training_args.learning_rate}")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Batch size: {training_args.per_device_train_batch_size}")
print(f"   Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"   Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✅ Training arguments configured!

📋 Config B Settings:
   Learning rate: 0.0001
   Epochs: 3
   Batch size: 4
   Gradient accumulation: 4
   Effective batch size: 16


In [ ]:
# Cell 8: Train the Model!
from transformers import Trainer
import wandb

# Initialize W&B run
wandb.init(
    project="sql-rag-finetuning",
    name="config-b-lr1e4-r16",
    config={
        "learning_rate": 1e-4,
        "lora_r": 16,
        "lora_alpha": 32,
        "epochs": 3,
        "batch_size": 4,
        "model": "Mistral-7B-Instruct-v0.2"
    }
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
)

print("🚀 Starting training...")
print("   This will take ~45-60 minutes on T4 GPU")
print("   You can monitor progress at: https://wandb.ai\n")

# Train!
trainer.train()

print("\n✅ Training complete!")
```

**Run this now!** This is the main training - takes ~45-60 min. ☕

You'll see progress like:
```
Step 25/1312 | Loss: 1.234 | ...
Step 50/1312 | Loss: 0.987 | ...

SyntaxError: invalid character '☕' (U+2615) (ipython-input-2679977969.py, line 38)

In [ ]:
# Cell 8: Train the Model!
from transformers import Trainer
import wandb

# Initialize W&B run
wandb.init(
    project="sql-rag-finetuning",
    name="config-b-lr1e4-r16",
    config={
        "learning_rate": 1e-4,
        "lora_r": 16,
        "lora_alpha": 32,
        "epochs": 3,
        "batch_size": 4,
        "model": "Mistral-7B-Instruct-v0.2"
    }
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
)

print("🚀 Starting training...")
print("   This will take ~45-60 minutes on T4 GPU")
print("   You can monitor progress at: https://wandb.ai\n")

# Train!
trainer.train()

print("\n✅ Training complete!")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aravind-b25 (aravind-b25-northeastern-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


NameError: name 'model' is not defined

In [ ]:
# Cell: Reload and Train (All-in-one)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_from_disk
import wandb

# ============================================================
# 1. LOAD MODEL
# ============================================================
print("🔄 Loading Mistral-7B with 4-bit quantization...")

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
print(f"✅ Model loaded!")

# ============================================================
# 2. APPLY LORA
# ============================================================
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ============================================================
# 3. LOAD & TOKENIZE DATASET
# ============================================================
print("\n🔄 Loading dataset...")
dataset = load_from_disk('data/processed/sql_dataset')

def tokenize_function(examples):
    result = tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset["train"].column_names)
print(f"✅ Dataset ready: {len(tokenized_dataset['train'])} training samples")

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# ============================================================
# 4. TRAINING ARGUMENTS
# ============================================================
training_args = TrainingArguments(
    output_dir="./models/config_b",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    optim="paged_adamw_8bit",
    logging_steps=25,
    report_to="wandb",
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    label_names=["labels"]
)

# ============================================================
# 5. TRAIN!
# ============================================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
)

print("\n🚀 Starting training... (45-60 min on T4)")
print("📊 Monitor at: https://wandb.ai/aravind-b25-northeastern-university/sql-rag-finetuning\n")

trainer.train()

print("\n✅ Training complete!")

🔄 Loading Mistral-7B with 4-bit quantization...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [ ]:
# Reinstall dependencies
!pip install -q bitsandbytes>=0.46.1 accelerate peft transformers datasets

print("✅ Dependencies installed!")

✅ Dependencies installed!


In [ ]:
# Cell: Reload and Train (All-in-one)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_from_disk
import wandb

# ============================================================
# 1. LOAD MODEL
# ============================================================
print("🔄 Loading Mistral-7B with 4-bit quantization...")

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
print(f"✅ Model loaded!")

# ============================================================
# 2. APPLY LORA
# ============================================================
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ============================================================
# 3. LOAD & TOKENIZE DATASET
# ============================================================
print("\n🔄 Loading dataset...")
dataset = load_from_disk('data/processed/sql_dataset')

def tokenize_function(examples):
    result = tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset["train"].column_names)
print(f"✅ Dataset ready: {len(tokenized_dataset['train'])} training samples")

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# ============================================================
# 4. TRAINING ARGUMENTS
# ============================================================
training_args = TrainingArguments(
    output_dir="./models/config_b",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    optim="paged_adamw_8bit",
    logging_steps=25,
    report_to="wandb",
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    label_names=["labels"]
)

# ============================================================
# 5. TRAIN!
# ============================================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
)

print("\n🚀 Starting training... (45-60 min on T4)")
print("📊 Monitor at: https://wandb.ai\n")

trainer.train()

print("\n✅ Training complete!")

🔄 Loading Mistral-7B with 4-bit quantization...


ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [ ]:
# COMPLETE SETUP + TRAINING (Run after restart)

# Step 1: Install packages
!pip install -q bitsandbytes>=0.46.1 accelerate peft transformers datasets wandb

# Step 2: Imports
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_from_disk, load_dataset, Dataset, DatasetDict
from sklearn.model_selection import train_test_split
import wandb
import os

print(f"✅ CUDA available: {torch.cuda.is_available()}")
print(f"✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

# Step 3: Recreate dataset if needed
os.makedirs('data/processed', exist_ok=True)
if not os.path.exists('data/processed/sql_dataset'):
    print("\n🔄 Recreating dataset...")
    spider = load_dataset("spider")
    def format_example(ex):
        prompt = f"""<s>[INST] You are an expert SQL assistant. Given a database schema and a natural language question, generate the correct SQL query. Only output the SQL query, nothing else.

Database: {ex['db_id']}

Question: {ex['question']} [/INST]
{ex['query']}</s>"""
        return {'text': prompt, 'question': ex['question'], 'query': ex['query'], 'db_id': ex['db_id']}

    train_fmt = [format_example(ex) for ex in spider['train']]
    val_data = list(spider['validation'])
    val_split, test_split = train_test_split(val_data, test_size=0.5, random_state=42)
    dataset = DatasetDict({
        'train': Dataset.from_list(train_fmt),
        'validation': Dataset.from_list([format_example(ex) for ex in val_split]),
        'test': Dataset.from_list([format_example(ex) for ex in test_split])
    })
    dataset.save_to_disk('data/processed/sql_dataset')

dataset = load_from_disk('data/processed/sql_dataset')
print(f"✅ Dataset: {len(dataset['train'])} train, {len(dataset['validation'])} val")

# Step 4: Load Model
print("\n🔄 Loading Mistral-7B (4-bit)...")
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)
print("✅ Model loaded!")

# Step 5: Apply LoRA
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Step 6: Tokenize
def tokenize_fn(examples):
    result = tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")
    result["labels"] = result["input_ids"].copy()
    return result

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=dataset["train"].column_names)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Step 7: W&B Login
wandb.login()

# Step 8: Training Args
training_args = TrainingArguments(
    output_dir="./models/config_b",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    optim="paged_adamw_8bit",
    logging_steps=25,
    report_to="wandb",
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    fp16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    label_names=["labels"]
)

# Step 9: Train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
)

print("\n🚀 Starting training... (~45-60 min on T4)")
trainer.train()
print("\n✅ Training complete!")

✅ CUDA available: True
✅ GPU: Tesla T4

🔄 Recreating dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

spider/train-00000-of-00001.parquet:   0%|          | 0.00/831k [00:00<?, ?B/s]

spider/validation-00000-of-00001.parquet:   0%|          | 0.00/126k [00:00<?, ?B/s]

...

Map:   0%|          | 0/517 [00:00<?, ? examples/s]

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: aravind-b25 (aravind-b25-northeastern-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



🚀 Starting training... (~45-60 min on T4)


Step,Training Loss,Validation Loss
100,0.831459,0.792969
200,0.506516,0.678777


KeyboardInterrupt: 

In [ ]:
# FRESH START WITH PHI-3-MINI (Faster Training!)

# Install
!pip install -q bitsandbytes>=0.46.1 accelerate peft transformers datasets wandb

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_from_disk
import wandb

print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

# ============================================================
# 1. LOAD PHI-3-MINI (Much faster!)
# ============================================================
print("\n🔄 Loading Phi-3-mini (3.8B params)...")
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
print(f"✅ Model loaded! Memory: {model.get_memory_footprint() / 1e9:.2f} GB")

# ============================================================
# 2. APPLY LORA
# ============================================================
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["qkv_proj", "o_proj"],  # Phi-3 uses different names
    lora_dropout=0.1, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ============================================================
# 3. LOAD DATASET
# ============================================================
dataset = load_from_disk('data/processed/sql_dataset')
print(f"✅ Dataset: {len(dataset['train'])} samples")

def tokenize_fn(examples):
    result = tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")
    result["labels"] = result["input_ids"].copy()
    return result

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=dataset["train"].column_names)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# ============================================================
# 4. TRAINING (Faster settings)
# ============================================================
wandb.init(project="sql-rag-finetuning", name="phi3-mini-config-b")

training_args = TrainingArguments(
    output_dir="./models/phi3_config_b",
    num_train_epochs=2,  # 2 epochs is enough for Phi-3
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,  # Slightly higher LR for smaller model
    weight_decay=0.01,
    warmup_ratio=0.1,
    optim="paged_adamw_8bit",
    logging_steps=25,
    report_to="wandb",
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    fp16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    label_names=["labels"]
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
)

print("\n🚀 Starting training with Phi-3-mini...")
print("⏱️  Estimated time: ~1.5-2 hours\n")

trainer.train()

# Save the model
trainer.save_model("./models/phi3_config_b/final")
print("\n✅ Training complete! Model saved.")
wandb.finish()

✅ GPU: Tesla T4

🔄 Loading Phi-3-mini (3.8B params)...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

modeling_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

KeyError: 'type'

In [ ]:
# GEMMA-2B (GOOGLE) - Best quality + reasonable time

!pip install -q bitsandbytes>=0.46.1 accelerate peft transformers datasets wandb

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_from_disk
import wandb

print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

# ============================================================
# 1. LOAD GEMMA-2B (Google)
# ============================================================
print("\n🔄 Loading Gemma-2B (Google DeepMind)...")
MODEL_NAME = "google/gemma-2b-it"  # instruction-tuned version

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)
print(f"✅ Model loaded! Memory: {model.get_memory_footprint() / 1e9:.2f} GB")

# ============================================================
# 2. APPLY LORA
# ============================================================
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ============================================================
# 3. LOAD DATASET
# ============================================================
dataset = load_from_disk('data/processed/sql_dataset')
print(f"✅ Dataset: {len(dataset['train'])} samples")

def tokenize_fn(examples):
    result = tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")
    result["labels"] = result["input_ids"].copy()
    return result

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=dataset["train"].column_names)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# ============================================================
# 4. TRAINING
# ============================================================
wandb.init(project="sql-rag-finetuning", name="gemma2b-config-b")

training_args = TrainingArguments(
    output_dir="./models/gemma2b_config_b",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    optim="paged_adamw_8bit",
    logging_steps=50,
    report_to="wandb",
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    fp16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    label_names=["labels"]
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
)

print("\n🚀 Starting training with Gemma-2B (Google)...")
print("⏱️  Estimated time: ~1 hour\n")

trainer.train()

# Save the model
trainer.save_model("./models/gemma2b_config_b/final")
print("\n✅ Training complete! Model saved.")
wandb.finish()

✅ GPU: Tesla T4

🔄 Loading Gemma-2B (Google DeepMind)...


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/gemma-2b-it.
401 Client Error. (Request ID: Root=1-698164cb-690f8e8c7cf4dee46aeab268;21058eb0-a5d8-456b-a5d4-246fdcd6065e)

Cannot access gated repo for url https://huggingface.co/google/gemma-2b-it/resolve/main/config.json.
Access to model google/gemma-2b-it is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
# TINYLLAMA - FAST & NO RESTRICTIONS (~30-45 min)

!pip install -q bitsandbytes>=0.46.1 accelerate peft transformers datasets wandb

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_from_disk
import wandb

print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

# ============================================================
# 1. LOAD TINYLLAMA
# ============================================================
print("\n🔄 Loading TinyLlama-1.1B...")
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)
print(f"✅ Model loaded! Memory: {model.get_memory_footprint() / 1e9:.2f} GB")

# ============================================================
# 2. APPLY LORA
# ============================================================
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ============================================================
# 3. LOAD DATASET
# ============================================================
dataset = load_from_disk('data/processed/sql_dataset')
print(f"✅ Dataset: {len(dataset['train'])} samples")

def tokenize_fn(examples):
    result = tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")
    result["labels"] = result["input_ids"].copy()
    return result

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=dataset["train"].column_names)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# ============================================================
# 4. TRAINING
# ============================================================
wandb.init(project="sql-rag-finetuning", name="tinyllama-config-b")

training_args = TrainingArguments(
    output_dir="./models/tinyllama_config_b",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    optim="paged_adamw_8bit",
    logging_steps=50,
    report_to="wandb",
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    fp16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    label_names=["labels"]
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
)

print("\n🚀 Starting training with TinyLlama...")
print("⏱️  Estimated time: ~30-45 minutes\n")

trainer.train()

# Save the model
trainer.save_model("./models/tinyllama_config_b/final")
print("\n✅ Training complete! Model saved.")
wandb.finish()

✅ GPU: Tesla T4

🔄 Loading TinyLlama-1.1B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Model loaded! Memory: 0.75 GB
trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


FileNotFoundError: Directory data/processed/sql_dataset not found

In [ ]:
# COMPLETE ALL-IN-ONE: Dataset + TinyLlama Training

!pip install -q bitsandbytes>=0.46.1 accelerate peft transformers datasets wandb scikit-learn

import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset, load_from_disk, Dataset, DatasetDict
from sklearn.model_selection import train_test_split
import wandb

print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

# ============================================================
# 1. PREPARE DATASET (if not exists)
# ============================================================
os.makedirs('data/processed', exist_ok=True)

if not os.path.exists('data/processed/sql_dataset'):
    print("\n🔄 Preparing dataset...")
    spider = load_dataset("spider")

    def format_example(ex):
        prompt = f"""<s>[INST] You are an expert SQL assistant. Generate the correct SQL query.

Database: {ex['db_id']}

Question: {ex['question']} [/INST]
{ex['query']}</s>"""
        return {'text': prompt, 'question': ex['question'], 'query': ex['query'], 'db_id': ex['db_id']}

    train_fmt = [format_example(ex) for ex in spider['train']]
    val_data = list(spider['validation'])
    val_split, test_split = train_test_split(val_data, test_size=0.5, random_state=42)

    dataset = DatasetDict({
        'train': Dataset.from_list(train_fmt),
        'validation': Dataset.from_list([format_example(ex) for ex in val_split]),
        'test': Dataset.from_list([format_example(ex) for ex in test_split])
    })
    dataset.save_to_disk('data/processed/sql_dataset')
    print("✅ Dataset created!")
else:
    dataset = load_from_disk('data/processed/sql_dataset')
    print("✅ Dataset loaded from disk!")

print(f"   Train: {len(dataset['train'])}, Val: {len(dataset['validation'])}, Test: {len(dataset['test'])}")

# ============================================================
# 2. LOAD TINYLLAMA
# ============================================================
print("\n🔄 Loading TinyLlama-1.1B...")
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)
print(f"✅ Model loaded! Memory: {model.get_memory_footprint() / 1e9:.2f} GB")

# ============================================================
# 3. APPLY LORA
# ============================================================
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ============================================================
# 4. TOKENIZE
# ============================================================
def tokenize_fn(examples):
    result = tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")
    result["labels"] = result["input_ids"].copy()
    return result

print("\n🔄 Tokenizing...")
tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=dataset["train"].column_names)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print("✅ Tokenization complete!")

# ============================================================
# 5. W&B + TRAINING
# ============================================================
wandb.init(project="sql-rag-finetuning", name="tinyllama-config-b")

training_args = TrainingArguments(
    output_dir="./models/tinyllama_config_b",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    optim="paged_adamw_8bit",
    logging_steps=50,
    report_to="wandb",
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    fp16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    label_names=["labels"]
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
)

print("\n🚀 Starting training with TinyLlama...")
print("⏱️  Estimated time: ~30-45 minutes\n")

trainer.train()

# Save
trainer.save_model("./models/tinyllama_config_b/final")
tokenizer.save_pretrained("./models/tinyllama_config_b/final")
print("\n✅ Training complete! Model saved.")
wandb.finish()

✅ GPU: Tesla T4

🔄 Preparing dataset...


README.md: 0.00B [00:00, ?B/s]

spider/train-00000-of-00001.parquet:   0%|          | 0.00/831k [00:00<?, ?B/s]

spider/validation-00000-of-00001.parquet:   0%|          | 0.00/126k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

...

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



🚀 Starting training with TinyLlama...
⏱️  Estimated time: ~30-45 minutes



Step,Training Loss,Validation Loss
100,1.010648,0.895109
200,0.733936,0.868105
300,0.651827,0.862977
400,0.606803,0.903659
500,0.565401,0.917573
600,0.531653,0.924732
700,0.500389,0.934314
800,0.494446,0.981262
900,0.458614,0.997954
1000,0.448006,1.003929



✅ Training complete! Model saved.


eval/loss,▂▁▁▃▃▄▄▆▇▇▇██
eval/runtime,▃▅▄▄▄▃▄█▄▃▃▅▁
eval/samples_per_second,▆▄▅▅▅▆▅▁▅▆▆▄█
eval/steps_per_second,▆▃▅▆▆▇▆▁▆▆▆▃█
train/epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
train/global_step,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
train/grad_norm,▂▁▁▂▁▁▁▁▁▁▁▁▂▃▁▃▁▁▃█▁▃▅▄▂▂
train/learning_rate,▄▆██▇▇▇▆▆▆▆▅▅▅▄▄▄▃▃▃▃▂▂▂▁▁
train/loss,█▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/loss,1.01775
eval/runtime,45.4706


In [ ]:
# CONFIG A: lr=3e-4, r=8 (Higher LR, smaller LoRA rank)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_from_disk
import wandb

print("🔄 Loading model for Config A...")
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto")
model = prepare_model_for_kbit_training(model)

# CONFIG A: r=8, lr=3e-4
lora_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Load dataset
dataset = load_from_disk('data/processed/sql_dataset')
def tokenize_fn(examples):
    result = tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")
    result["labels"] = result["input_ids"].copy()
    return result
tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=dataset["train"].column_names)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# W&B
wandb.init(project="sql-rag-finetuning", name="tinyllama-config-a-r8-lr3e4")

# 1 epoch only
training_args = TrainingArguments(
    output_dir="./models/tinyllama_config_a",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=3e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    optim="paged_adamw_8bit",
    logging_steps=50,
    report_to="wandb",
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    label_names=["labels"]
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
)

print("\n🚀 Training Config A (lr=3e-4, r=8, 1 epoch)...")
print("⏱️  ~20 minutes\n")
trainer.train()

trainer.save_model("./models/tinyllama_config_a/final")
print("\n✅ Config A complete!")
wandb.finish()

🔄 Loading model for Config A...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/517 [00:00<?, ? examples/s]

...

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



🚀 Training Config A (lr=3e-4, r=8, 1 epoch)...
⏱️  ~20 minutes



Epoch,Training Loss,Validation Loss
1,0.624216,0.891884



✅ Config A complete!


eval/loss,▁
eval/runtime,▁
eval/samples_per_second,▁
eval/steps_per_second,▁
train/epoch,▁▂▃▄▅▆▆▇██
train/global_step,▁▂▃▄▅▆▆▇██
train/grad_norm,█▁▁▄▃▃▃▅
train/learning_rate,█▇▆▅▄▃▂▁
train/loss,█▂▂▂▁▁▁▁
eval/loss,0.89188
eval/runtime,45.6257


In [ ]:
# CONFIG C: lr=1e-4, r=32 (Lower LR, larger LoRA rank)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_from_disk
import wandb

print("🔄 Loading model for Config C...")
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto")
model = prepare_model_for_kbit_training(model)

# CONFIG C: r=32, lr=1e-4
lora_config = LoraConfig(
    r=32, lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Load dataset
dataset = load_from_disk('data/processed/sql_dataset')
def tokenize_fn(examples):
    result = tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")
    result["labels"] = result["input_ids"].copy()
    return result
tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=dataset["train"].column_names)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# W&B
wandb.init(project="sql-rag-finetuning", name="tinyllama-config-c-r32-lr1e4")

# 1 epoch only
training_args = TrainingArguments(
    output_dir="./models/tinyllama_config_c",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    optim="paged_adamw_8bit",
    logging_steps=50,
    report_to="wandb",
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    label_names=["labels"]
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
)

print("\n🚀 Training Config C (lr=1e-4, r=32, 1 epoch)...")
print("⏱️  ~25 minutes\n")
trainer.train()

trainer.save_model("./models/tinyllama_config_c/final")
print("\n✅ Config C complete!")
wandb.finish()

🔄 Loading model for Config C...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

trainable params: 9,011,200 || all params: 1,109,059,584 || trainable%: 0.8125


Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/517 [00:00<?, ? examples/s]

...

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



🚀 Training Config C (lr=1e-4, r=32, 1 epoch)...
⏱️  ~25 minutes



Epoch,Training Loss,Validation Loss
1,0.639845,0.888985



✅ Config C complete!


eval/loss,▁
eval/runtime,▁
eval/samples_per_second,▁
eval/steps_per_second,▁
train/epoch,▁▂▃▄▅▆▆▇██
train/global_step,▁▂▃▄▅▆▆▇██
train/grad_norm,█▁▁▂▂▂▂▃
train/learning_rate,█▇▆▅▄▃▂▁
train/loss,█▂▂▁▁▁▁▁
eval/loss,0.88898
eval/runtime,45.6598


In [ ]:
# Save models to Google Drive
from google.colab import drive
drive.mount('/content/drive')

!cp -r ./models /content/drive/MyDrive/sql_rag_models
!cp -r ./data /content/drive/MyDrive/sql_rag_data

print("✅ Models and data saved to Google Drive!")

Mounted at /content/drive
✅ Models and data saved to Google Drive!
